In [ ]:
import torch 
import numpy as np 
import matplotlib.pyplot as plt
from itertools import islice
import json
from tqdm.auto import tqdm

from cryo_sbi.wpa_simulator.cryo_em_simulator import cryo_em_simulator
from cryo_sbi import CryoEmSimulator
from cryo_sbi.inference.models import build_models
import cryo_sbi.utils.estimator_utils as est_utils
from cryo_sbi.inference.priors import get_image_priors, PriorLoader
from cryo_sbi.inference.models.build_models import build_nle_flow_model

In [ ]:
# Parameters
device='cuda'
# load models and images
models = torch.load("models.pt").to(device)
images = torch.load("images.pt").to(device)

# Usage:
# Load trained model
estimator = est_utils.load_estimator(
    "training_parameters_nle.json",
    build_models.build_nle_flow_model,
    "tutorial_estimator.pt",
    device="cuda",
)
estimator.eval()
estimator.to(device)

In [ ]:
# check if theta embedding not collapsed
def check_theta_embedding_quality(
    estimator: torch.nn.Module,
    models: torch.Tensor,
    device: str = "cuda"
):
    """
    Check if theta embedding (MLP) has collapsed after training.
    
    Args:
        estimator: Trained NLE estimator
        models: Conformational models [num_models, 3, num_beads] or [num_models, num_beads, 3]
        device: Device to use
        
    Returns:
        dict: Dictionary with embedding quality metrics
    """
    
    # Ensure models are in correct format [num_models, num_beads, 3]
    if models.shape[1] == 3 and models.shape[2] != 3:
        models = models.transpose(1, 2)
    
    models = models.to(device)
    num_models = len(models)
    
    # Extract theta embedding network
    theta_embedding = estimator.embedding_theta
    theta_embedding.eval()
    
    print("="*60)
    print("THETA EMBEDDING QUALITY CHECK")
    print("="*60)
    print(f"Number of models: {num_models}")
    print(f"Model shape: {models.shape}")
    
    # Compute embeddings for all models
    with torch.no_grad():
        embeddings = theta_embedding(models)
    
    print(f"Embedding shape: {embeddings.shape}")
    print(f"Embedding dimension: {embeddings.shape[1]}")
    
    # Compute diversity metrics
    emb_mean = embeddings.mean().item()
    emb_std = embeddings.std().item()
    emb_std_across_models = embeddings.std(dim=0).mean().item()
    emb_magnitude = embeddings.abs().mean().item()
    
    print(f"\n{'Embedding Statistics':^60}")
    print("-"*60)
    print(f"Mean:                     {emb_mean:>10.4f}")
    print(f"Overall Std:              {emb_std:>10.4f}")
    print(f"Std across models:        {emb_std_across_models:>10.4f}")
    print(f"Mean magnitude:           {emb_magnitude:>10.4f}")
    
    # Compute pairwise distances
    pairwise_dist = torch.cdist(embeddings, embeddings)
    pairwise_dist_no_diag = pairwise_dist.clone()
    pairwise_dist_no_diag.fill_diagonal_(float('inf'))
    
    dist_mean = pairwise_dist.mean().item()
    dist_min = pairwise_dist_no_diag.min().item()
    dist_max = pairwise_dist.max().item()
    dist_std = pairwise_dist.std().item()
    
    print(f"\n{'Pairwise Distances':^60}")
    print("-"*60)
    print(f"Mean:                     {dist_mean:>10.4f}")
    print(f"Min (excluding diagonal): {dist_min:>10.4f}")
    print(f"Max:                      {dist_max:>10.4f}")
    print(f"Std:                      {dist_std:>10.4f}")
    
    # Print some example embeddings
    print(f"\n{'Sample Embeddings (first 5 models, first 10 dims)':^60}")
    print("-"*60)
    for i in range(min(5, num_models)):
        emb_sample = embeddings[i, :10].cpu().numpy()
        print(f"Model {i}: [{', '.join([f'{x:7.4f}' for x in emb_sample])}...]")
    
    # Assess quality
    print(f"\n{'Quality Assessment':^60}")
    print("-"*60)
    
    # Check 1: Std across models should be > 0.1 (good diversity)
    if emb_std_across_models > 0.1:
        print(f"✓ PASS: Std across models = {emb_std_across_models:.4f} > 0.1")
        std_status = "PASS"
    elif emb_std_across_models > 0.01:
        print(f"⚠ WARNING: Std across models = {emb_std_across_models:.4f} is low")
        std_status = "WARNING"
    else:
        print(f"✗ FAIL: Std across models = {emb_std_across_models:.4f} < 0.01 (COLLAPSED!)")
        std_status = "FAIL"
    
    # Check 2: Min pairwise distance should be > 0.5 (well-separated)
    if dist_min > 0.5:
        print(f"✓ PASS: Min pairwise distance = {dist_min:.4f} > 0.5")
        dist_status = "PASS"
    elif dist_min > 0.1:
        print(f"⚠ WARNING: Min pairwise distance = {dist_min:.4f} is small")
        dist_status = "WARNING"
    else:
        print(f"✗ FAIL: Min pairwise distance = {dist_min:.4f} < 0.1 (COLLAPSED!)")
        dist_status = "FAIL"
    
    # Check 3: Mean pairwise distance should be substantial
    if dist_mean > 2.0:
        print(f"✓ PASS: Mean pairwise distance = {dist_mean:.4f} > 2.0")
        mean_dist_status = "PASS"
    elif dist_mean > 1.0:
        print(f"⚠ WARNING: Mean pairwise distance = {dist_mean:.4f} is moderate")
        mean_dist_status = "WARNING"
    else:
        print(f"✗ FAIL: Mean pairwise distance = {dist_mean:.4f} < 1.0 (COLLAPSED!)")
        mean_dist_status = "FAIL"
    
    # Overall assessment
    print(f"\n{'Overall Status':^60}")
    print("-"*60)
    if all(status == "PASS" for status in [std_status, dist_status, mean_dist_status]):
        print("✓✓✓ EXCELLENT: Embeddings are highly discriminative!")
        overall_status = "EXCELLENT"
    elif all(status in ["PASS", "WARNING"] for status in [std_status, dist_status, mean_dist_status]):
        print("✓✓  GOOD: Embeddings are discriminative but could be better")
        overall_status = "GOOD"
    elif any(status == "FAIL" for status in [std_status, dist_status, mean_dist_status]):
        print("✗✗✗ POOR: Embeddings have COLLAPSED or are not discriminative!")
        overall_status = "POOR"
    else:
        print("⚠⚠  MARGINAL: Embeddings may not be optimal")
        overall_status = "MARGINAL"
    
    print("="*60 + "\n")
    
    # Return metrics as dictionary
    return {
        'embeddings': embeddings.cpu(),
        'mean': emb_mean,
        'std': emb_std,
        'std_across_models': emb_std_across_models,
        'magnitude': emb_magnitude,
        'pairwise_dist_mean': dist_mean,
        'pairwise_dist_min': dist_min,
        'pairwise_dist_max': dist_max,
        'pairwise_dist_std': dist_std,
        'std_status': std_status,
        'dist_status': dist_status,
        'mean_dist_status': mean_dist_status,
        'overall_status': overall_status
    }


# run checks
# Check embedding quality
metrics = check_theta_embedding_quality(estimator=estimator, models=models, device="cuda")

In [ ]:
def evaluate_likelihood_matrix_efficient(
    estimator, 
    images, 
    models, 
    image_batch_size=8,
    model_batch_size=32, 
    device='cuda',
    show_progress=True
):
    """
    More efficient version that batches both images and models.
    
    Args:
        estimator: Trained NLEWithEmbedding model
        images: All images [num_images, H, W]
        models: All conformations [num_models, 3, N]
        image_batch_size: Number of images to process together
        model_batch_size: Number of models to process together
        device: Device to use
        show_progress: Whether to show progress bar
    
    Returns:
        log_likelihood_matrix: [num_images, num_models] 
    """
    
    # Ensure images have correct shape
    if images.dim() == 2:
        images = images.unsqueeze(0)
    
    num_images = images.shape[0]
    num_models = models.shape[0]
    
    # Initialize result matrix
    log_likelihood_matrix = torch.zeros(num_images, num_models)
    
    # Create progress bar for total iterations
    total_iterations = ((num_images + image_batch_size - 1) // image_batch_size) * \
                       ((num_models + model_batch_size - 1) // model_batch_size)
    
    pbar = tqdm(total=total_iterations, desc="Computing likelihood matrix") if show_progress else None
    
    with torch.no_grad():
        # Iterate over image batches
        for img_start in range(0, num_images, image_batch_size):
            img_end = min(img_start + image_batch_size, num_images)
            batch_images = images[img_start:img_end].to(device)  # [B_img, H, W]
            curr_img_batch = batch_images.shape[0]
            
            # Iterate over model batches
            for model_start in range(0, num_models, model_batch_size):
                model_end = min(model_start + model_batch_size, num_models)
                batch_models = models[model_start:model_end].to(device)  # [B_model, 3, N]
                curr_model_batch = batch_models.shape[0]
                
                # Transpose for GNN: [B_model, 3, N] -> [B_model, N, 3]
                batch_models = batch_models.transpose(1, 2)
                
                # Create all pairs: repeat images and models
                # Each image paired with each model in the batch
                expanded_images = batch_images.unsqueeze(1).repeat(1, curr_model_batch, 1, 1)  
                # [B_img, B_model, H, W]
                expanded_images = expanded_images.reshape(-1, batch_images.shape[1], batch_images.shape[2])  
                # [B_img * B_model, H, W]
                
                expanded_models = batch_models.unsqueeze(0).repeat(curr_img_batch, 1, 1, 1)  
                # [B_img, B_model, N, 3]
                expanded_models = expanded_models.reshape(-1, batch_models.shape[1], batch_models.shape[2])  
                # [B_img * B_model, N, 3]
                
                # Compute log likelihood for all pairs
                log_probs = estimator.forward(expanded_images, expanded_models)  # [B_img * B_model]
                
                # Reshape back to matrix form
                log_probs = log_probs.reshape(curr_img_batch, curr_model_batch)  # [B_img, B_model]
                
                # Store in result matrix
                log_likelihood_matrix[img_start:img_end, model_start:model_end] = log_probs.cpu()
                
                if pbar:
                    pbar.update(1)
    
    if pbar:
        pbar.close()
    
    return log_likelihood_matrix

log_likelihood_matrix = evaluate_likelihood_matrix_efficient(
    estimator, 
    images,  # [num_images, H, W]
    models,  # [num_models, 3, N]
    image_batch_size=100,   # Process 8 images at once
    model_batch_size=20,  # Process 32 models at once
    device='cuda',
    show_progress=True
)

In [ ]:
np.save('log_prob.npy', log_likelihood_matrix.numpy())